In [1]:
"""
The purpose of this Jupyter notebook is to re-compute the Z'-factors
after multiplication with the PU model probabilities.

The purpose of doing so is to evaluate whether this refinement renders
the screen analyzable.
"""

"\nThe purpose of this Jupyter notebook is to re-compute the Z'-factors\nafter multiplication with the PU model probabilities.\n\nThe purpose of doing so is to evaluate whether this refinement renders\nthe screen analyzable.\n"

In [2]:
import pandas as pd

# Loading and Processing Data

In [3]:
# Load the screen TSV file into a Pandas DataFrame
screen_df = pd.read_csv(
    "/Users/jacobanter/Documents/Code/VACV_screen/Dharmacon_pooled_"
    "genome_1_and_2_subset_with_control-based_Z_scoring/Dharmacon_"
    "pooled_G1_G2_screening_plates_subset_with_missing_UniProt_IDs_"
    "control-based_Z-scored.tsv",
    sep="\t"
)

In [4]:
# Determine the unique controls used in the screen
controls = screen_df.loc[
    screen_df["WellType"] == "CONTROL",
    "ID_openBIS"
].unique()

In [5]:
print(controls)

['ATP6V1A' 'SCRAMBLED' 'MOCK' 'PSMC3' 'KIF11' 'PSMA6' 'TSG101' 'GFP'
 'RAC1' 'ARPC3' 'CDC42' 'PAK1' 'UNKNOWN']


In [6]:
# Define positive controls
pos_controls = [
    control
    for control in controls
    if (control != "UNKNOWN")
    and
    (control != "SCRAMBLED")
    and
    (control != "MOCK")
]

In [7]:
print(pos_controls)

['ATP6V1A', 'PSMC3', 'KIF11', 'PSMA6', 'TSG101', 'GFP', 'RAC1', 'ARPC3', 'CDC42', 'PAK1']


In [8]:
# Load the probabilities outputted by the PU learning model
pu_probs_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Multimodal_achitecture_inference/sample-specific_"
    "modality_gating/inference_results_only_PPI_features/all_hf_entire_"
    "screen_predictions_PPI_only_model.tsv"
)

pu_probs_df = pd.read_csv(
    pu_probs_path,
    sep="\t"
)

# Multiplication of Intensities by the PU Model Probabilities

In [9]:
# First, extract the columns of interest from the screen DataFrame
intensity_df = screen_df.copy()

In [10]:
# Create a mapping from gene to probability
prob_map = pu_probs_df.set_index("gene")["probability"]

# Look up the probability for each row
prob = intensity_df["Name"].map(prob_map).fillna(1.0)

# Multiply the desired columns
cols = [
    "dIntensity_cPathogen_eMean_oVoronoiCells",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells"
]

intensity_df[cols] = intensity_df[cols].mul(prob, axis=0)

# Defining a Function for Z'-Factor Computation

In [13]:
# For the sake of convenience, a function for computing the Z'-factors
# is defined
def compute_z_prime_factor(neg_control, pos_control, int_df, plate_id):
    """
    Computes the Z'-factors for a pair of negative and positive control.

    Specifically, one Z'-factor is computed for early and late
    intensities each using intensities derived from Voronoi-defined
    cellular regions.

    Parameters
    ----------
    neg_control: str
        A string denoting the negative control.
    pos_control: str
        A string denoting the positive control.
    int_df: Pandas DataFrame
        A Pandas DataFrame storing the intensity values.
    plate_id: str
        A string denoting the plate ID.

    Returns
    -------
    early_z_prime_factor: float
        The early intensity Z'-factor the given pair of negative and
        positive control.
    late_z_prime_factor: float
        The late intensity Z'-factor for the given pair of negative and
        positive control.
    """
    # As a first step, retrieve the mean and standard deviation of the
    # intensities
    neg_control_ints = int_df.loc[
        (int_df["WellType"] == "CONTROL")
        &
        (int_df["Barcode"] == plate_id)
        &
        (int_df["ID_openBIS"] == neg_control),
        [
            "dIntensity_cPathogen_eMean_oVoronoiCells",
            "dIntensity_cLatePathogen_eMean_oVoronoiCells"
        ]
    ]

    neg_early_mean, neg_late_mean = neg_control_ints.mean().to_numpy()
    neg_early_std, neg_late_std = neg_control_ints.std().to_numpy()

    pos_control_ints = int_df.loc[
        (int_df["WellType"] == "CONTROL")
        &
        (int_df["Barcode"] == plate_id)
        &
        (int_df["ID_openBIS"] == pos_control),
        [
            "dIntensity_cPathogen_eMean_oVoronoiCells",
            "dIntensity_cLatePathogen_eMean_oVoronoiCells"
        ]
    ]

    # print(neg_control_ints.describe())
    # print(pos_control_ints.describe())

    pos_early_mean, pos_late_mean = pos_control_ints.mean().to_numpy()
    pos_early_std, pos_late_std = pos_control_ints.std().to_numpy()

    # Finally, compute the Z'-factors
    # The formula is as follows:
    # Z'-factor = 1 - \frac{3 * std_pos + 3 * std_neg}{mean_pos - mean_neg}
    z_prime_early = 1 - (
        (3 * pos_early_std + 3 * neg_early_std)
        /
        abs(pos_early_mean - neg_early_mean)
    )

    z_prime_late = 1 - (
        (3 * pos_late_std + 3 * neg_late_std)
        /
        abs(pos_late_mean - neg_late_mean)
    )
    
    return z_prime_early, z_prime_late

In [18]:
early, late = compute_z_prime_factor(
    "SCRAMBLED", "GFP", screen_df, "DZ01-2M"
)

print(early, late)

early, late = compute_z_prime_factor(
    "MOCK", "GFP", screen_df, "DZ01-2M"
)

print(early, late)

0.012294896541933054 -7.201808348449461
0.0304664894807809 -6.569898719237794


# Re-Computing the Z'-Factors

In [69]:
# Compute Z'-factors for SCRAMBLED as negative control and all positive
# controls
for pos_control in pos_controls:
    early, late = compute_z_prime_factor("SCRAMBLED", pos_control, intensity_df)
    print(
        f"Z'-factors for {pos_control} and SCRAMBLED: {early}, {late}"
    )

0.0610989716008772 0.013337881261314962
0.004185807578497544 0.0009165596085274128
Z'-factors for ATP6V1A and SCRAMBLED: 0.24862159143512852, 0.44634043585098315
0.0610989716008772 0.013337881261314962
0.00920641495627217 0.0020590451533150445
Z'-factors for PSMC3 and SCRAMBLED: 0.10987659443655096, 0.34691578377656773
0.0610989716008772 0.013337881261314962
0.0042841479854013155 0.0012876437883940987
Z'-factors for KIF11 and SCRAMBLED: 0.22772663264635096, 0.43222519617576793
0.0610989716008772 0.013337881261314962
0.0078005249989725875 0.0016956849844524974
Z'-factors for PSMA6 and SCRAMBLED: 0.15380838255629936, 0.36620916586163377
0.0610989716008772 0.013337881261314962
0.004903039148759804 0.0010733368385116258
Z'-factors for TSG101 and SCRAMBLED: 0.23066221320702074, 0.42738215932935153
0.0610989716008772 0.013337881261314962
0.043094042456140354 0.009205509997721878
Z'-factors for GFP and SCRAMBLED: -2.756203272639926, -152.7689661358618
0.0610989716008772 0.013337881261314962
0

In [70]:
# Compute Z'-factors for MOCK as negative control and all positive
# controls
for pos_control in pos_controls:
    early, late = compute_z_prime_factor("MOCK", pos_control, intensity_df)
    print(
        f"Z'-factors for {pos_control} and MOCK: {early}, {late}"
    )

0.061878032032163736 0.013318791249064104
0.004185807578497544 0.0009165596085274128
Z'-factors for ATP6V1A and MOCK: 0.25976068738565183, 0.4477740173522229
0.061878032032163736 0.013318791249064104
0.00920641495627217 0.0020590451533150445
Z'-factors for PSMC3 and MOCK: 0.124129621069612, 0.3491202899709922
0.061878032032163736 0.013318791249064104
0.0042841479854013155 0.0012876437883940987
Z'-factors for KIF11 and MOCK: 0.23916739012086363, 0.43375253194769225
0.061878032032163736 0.013318791249064104
0.0078005249989725875 0.0016956849844524974
Z'-factors for PSMA6 and MOCK: 0.1670579660245155, 0.36825624756525466
0.061878032032163736 0.013318791249064104
0.004903039148759804 0.0010733368385116258
Z'-factors for TSG101 and MOCK: 0.24218710564676704, 0.4289541144911999
0.061878032032163736 0.013318791249064104
0.043094042456140354 0.009205509997721878
Z'-factors for GFP and MOCK: -2.5973669739792995, -81.36570060863947
0.061878032032163736 0.013318791249064104
0.0038789508144751973 

# Computing Original Z'-Factors

In [13]:
# Also compute the Z'-factors obtained prior to the multiplication
for pos_control in pos_controls:
    early, late = compute_z_prime_factor(
        "SCRAMBLED", pos_control, screen_df, "DZ01-2M"
    )
    print(
        f"Z'-factors for {pos_control} and SCRAMBLED: {early}, {late}"
    )
    print()

Z'-factors for ATP6V1A and SCRAMBLED: -18.30322491791894, -649.3072306185586

Z'-factors for PSMC3 and SCRAMBLED: -1.3486810922284276, -3.3797973948408115

Z'-factors for KIF11 and SCRAMBLED: -92.17701403621722, -11.882525076705166

Z'-factors for PSMA6 and SCRAMBLED: -0.417774402591381, -2.551218142441092

Z'-factors for TSG101 and SCRAMBLED: -3.576940823956914, -28.452355107730444

Z'-factors for GFP and SCRAMBLED: 0.012294896541932832, -7.201808348449422

Z'-factors for RAC1 and SCRAMBLED: -2.1186060449690793, -4.833184636567825

Z'-factors for ARPC3 and SCRAMBLED: -4.439675998288729, -8.778151169128535

Z'-factors for CDC42 and SCRAMBLED: -5.647422735515478, -12.42867032030771

Z'-factors for PAK1 and SCRAMBLED: -7.003284618461274, -28.888663622267426



In [14]:
for pos_control in pos_controls:
    early, late = compute_z_prime_factor(
        "MOCK", pos_control, screen_df, "DZ01-2M"
    )
    print(
        f"Z'-factors for {pos_control} and MOCK: {early}, {late}"
    )
    print()

Z'-factors for ATP6V1A and MOCK: -35.607192689750356, -143.9588957873134

Z'-factors for PSMC3 and MOCK: -1.4386738840664672, -3.021965833079685

Z'-factors for KIF11 and MOCK: -236.06518862528767, -11.404733161355765

Z'-factors for PSMA6 and MOCK: -0.41335426495977545, -2.2209826665216545

Z'-factors for TSG101 and MOCK: -4.101139844464403, -30.910940958638605

Z'-factors for GFP and MOCK: 0.030466489480780456, -6.56989871923779

Z'-factors for RAC1 and MOCK: -2.3125322675908646, -4.19210709713811

Z'-factors for ARPC3 and MOCK: -5.159637432413034, -8.240654379330497

Z'-factors for CDC42 and MOCK: -6.866347754686082, -12.002406881747676

Z'-factors for PAK1 and MOCK: -8.763308387937306, -31.285886151571134



# Identifying the Best Z'-Factor

In [14]:
# Previous investigations revealed that the best Z'-factors are obtained
# using anti-GFP as positive controls
# For the manuscript, the best Z'-factor is determined

# Determine unique plate IDs
plate_ids = screen_df.loc[
    screen_df["WellType"] == "CONTROL",
    "Barcode"
].unique()

# Iterate over the plate IDs and compute for each of them the Z'-factor
# involving anti-GFP as positive control and either SCRAMBLED or MOCK as
# negative control
scrambled_Z_prime_factors_early = []
scrambled_z_prime_factors_late = []

mock_z_prime_factors_early = []
mock_z_prime_factors_late = []

for plate_id in plate_ids:
    scrambled_early, scrambled_late = compute_z_prime_factor(
        "SCRAMBLED", "GFP", screen_df, plate_id
    )
    scrambled_Z_prime_factors_early.append(scrambled_early)
    scrambled_z_prime_factors_late.append(scrambled_late)

    mock_early, mock_late = compute_z_prime_factor(
        "MOCK", "GFP", screen_df, plate_id
    )
    mock_z_prime_factors_early.append(mock_early)
    mock_z_prime_factors_late.append(mock_late)

In [18]:
# Determine the optimal Z'-factors, i.e. the highest ones (it is not
# generally true that the higher a Z'-factor, the better it is; however,
# since we are facing the problem of extremely negative Z'-factors, we
# want the Z'-factors to be as high as possible)
print(
    "Maximum Z'-factor using SCRAMBLED as negative control\nand early "
    f"intensities: {max(scrambled_Z_prime_factors_early)}\n"
    "Maximum Z'-factor using SCRAMBLED as negative control\nand late "
    f"intensities: {max(scrambled_z_prime_factors_late)}\n\n"
    "Maximum Z'-factor using MOCK as negative control\nand early "
    f"intensities: {max(mock_z_prime_factors_early)}\n"
    "Maximum Z'-factor using MOCK as negative control\nand late "
    f"intensities: {max(mock_z_prime_factors_late)}"

)

Maximum Z'-factor using SCRAMBLED as negative control
and early intensities: 0.41000843807613296
Maximum Z'-factor using SCRAMBLED as negative control
and late intensities: -1.7521646658523746

Maximum Z'-factor using MOCK as negative control
and early intensities: 0.42810302376944465
Maximum Z'-factor using MOCK as negative control
and late intensities: -1.49326931569946


In [20]:
# The best Z'-factor is obtained using MOCK as negative control and
# early intensities; it is 0.428
# This value is reported in the manuscript
# Since GFP is not associated with any PPI probabilities, the Z'-factors
# are not re-computed